<a href="https://colab.research.google.com/github/guitorte/audio/blob/claude/audio-analysis-mobile-notebook-lpowl5/track-digest/notebooks/Digest_and_Crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵🔎 track-digest — resumo p/ IA + corte de stems

Pós-processa uma pasta `<track>/` gerada pelo **music-to-midi**:

1. **Digest** — um resumo curto e barato em tokens (melodia/harmonia/ritmo do MIDI +
   timbre do áudio) p/ colar num assistente de IA.
2. **Corte** — recorta **todos os stems** numa região e exporta `.wav`.

```
music-to-midi/output/<track>/   (entrada)
├── stems/   vocals.wav, drums.wav, ...
├── midi/    vocals.mid, drums.mid, ...
└── <track>.mid
        ├─► analysis/<track>_digest.txt   (resumo p/ IA)
        └─► crops/<stem>_<ini>-<fim>s.wav
```

## 1. Instalar dependências

Stack leve (só consome `.mid`/`.wav` já gerados). **Não precisa reiniciar o runtime.**

In [ ]:
!pip -q install librosa soundfile pretty_midi numpy scipy ipywidgets
print('OK — deps instaladas.')

## 2. Imports e clone do repositório

Em Colab faz um shallow clone do repo em `/content/audio` e coloca `track-digest` no path.

In [ ]:
import os, sys, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = 'claude/audio-analysis-mobile-notebook-lpowl5'
if IN_COLAB:
    REPO_DIR = '/content/audio'
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                               'https://github.com/guitorte/audio.git', REPO_DIR])
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

PKG_DIR = os.path.join(REPO_DIR, 'track-digest')
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)

from modules import (build_track_digest, render_digest, write_digest,
                     default_digest_path, crop_stems, probe_duration,
                     list_audio_files, resolve_track_paths)
from IPython.display import Audio, display, FileLink

print(f'Colab  : {IN_COLAB}')
print(f'Pacote : {PKG_DIR}')

## 3. Montar Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive montado.')
else:
    print('Fora do Colab — pulando mount.')

## 4. Parâmetros + escolher a track

Aponte `OUTPUT_ROOT` para a pasta de saída do music-to-midi (onde estão as pastas
`<track>/`). Rode a célula de parâmetros e depois use o **menu** para escolher a track.

In [ ]:
# @title Parâmetros (deixe em branco p/ usar os padrões)
OUTPUT_ROOT     = ''  # @param {type:'string'}
TIMBRE_FROM     = 'raw'  # @param ['raw', 'clean']
ANALYZE_SECONDS = 60  # @param {type:'number'}

if not OUTPUT_ROOT:
    OUTPUT_ROOT = ('/content/drive/MyDrive/music-to-midi/output'
                   if IN_COLAB else os.path.join(REPO_DIR, 'music-to-midi', 'output'))
print(f'Pasta de saída do music-to-midi : {OUTPUT_ROOT}')
print(f'Timbre a partir de              : {TIMBRE_FROM} stems')

In [ ]:
# Lista as tracks (subpastas com stems/) e mostra um menu. Re-rode após processar novas.
import ipywidgets as widgets

def list_tracks(root):
    if not os.path.isdir(root):
        return []
    out = []
    for name in sorted(os.listdir(root)):
        d = os.path.join(root, name)
        if os.path.isdir(d) and (os.path.isdir(os.path.join(d, 'stems'))
                                 or os.path.isdir(os.path.join(d, 'stems_clean'))):
            out.append(d)
    return out

tracks = list_tracks(OUTPUT_ROOT)
print(f'{len(tracks)} track(s) em {OUTPUT_ROOT}:')
for t in tracks:
    print('  -', os.path.basename(t))

if not tracks:
    track_picker = None
    print(f'\n⚠️  Nenhuma track. Rode o music-to-midi primeiro (saída em {OUTPUT_ROOT}).')
else:
    track_picker = widgets.Dropdown(
        options=[(os.path.basename(t), t) for t in tracks],
        description='Track:', layout=widgets.Layout(width='80%'))
    display(track_picker)

## 5. Gerar o digest (resumo p/ IA)

Gera o `DIGEST v1` e salva em `analysis/<track>_digest.txt`. **Copie o bloco abaixo e
cole no seu assistente de IA** — é curto de propósito.

In [ ]:
assert track_picker is not None, 'Escolha uma track na célula 4.'
TRACK_DIR = track_picker.value
print(f'Track: {os.path.basename(TRACK_DIR)}\n')

td = build_track_digest(TRACK_DIR, timbre_from=TIMBRE_FROM,
                        analyze_seconds=float(ANALYZE_SECONDS) or None)
digest_text = render_digest(td)
print(digest_text)

out_txt = write_digest(td, default_digest_path(TRACK_DIR))
print('salvo em:', out_txt)
display(FileLink(out_txt))

## 6. Cortar stems — escolher a região

Ajuste início/fim com os sliders, escolha um stem e toque **Prévia** p/ ouvir o trecho.

In [ ]:
stems_dir = resolve_track_paths(TRACK_DIR).audio_stems_dir('raw')
stem_files = list_audio_files(stems_dir)
dur = max((probe_duration(p) for p in stem_files), default=30.0)

start_slider = widgets.FloatSlider(value=0.0, min=0.0, max=dur, step=0.1,
                                   description='Início (s)', layout=widgets.Layout(width='90%'))
end_slider   = widgets.FloatSlider(value=min(15.0, dur), min=0.0, max=dur, step=0.1,
                                   description='Fim (s)', layout=widgets.Layout(width='90%'))
stem_dd = widgets.Dropdown(options=[(os.path.basename(p), p) for p in stem_files],
                           description='Prévia:', layout=widgets.Layout(width='80%'))
preview_btn = widgets.Button(description='▶ Prévia', button_style='info')
out = widgets.Output()

def _preview(_):
    with out:
        out.clear_output()
        a, b = start_slider.value, min(end_slider.value, probe_duration(stem_dd.value))
        if b <= a:
            print('Fim deve ser maior que o início.'); return
        import librosa
        y, sr = librosa.load(stem_dd.value, sr=22050, mono=True, offset=a, duration=b - a)
        print(f'{os.path.basename(stem_dd.value)}  {a:.2f}-{b:.2f}s')
        display(Audio(y, rate=sr))

preview_btn.on_click(_preview)
display(start_slider, end_slider, stem_dd, preview_btn, out)

## 7. Exportar os cortes (.wav) + download

Corta **todos os stems** na região escolhida e (em Colab) baixa tudo num único `.zip`.

In [ ]:
res = crop_stems(TRACK_DIR, start_slider.value, end_slider.value, to_mono=False)
print('\n=== Cortes ===')
for stem, p in res.crops.items():
    print(f'  {stem:10s} -> {p}')
    display(FileLink(p))
if res.skipped:
    print('Pulados:', ', '.join(res.skipped))

if IN_COLAB and res.crops:
    import shutil
    from google.colab import files
    base = os.path.join(os.path.dirname(res.out_dir),
                        f'{os.path.basename(TRACK_DIR)}_crops')
    zip_path = shutil.make_archive(base, 'zip', res.out_dir)
    print('\nzip:', zip_path)
    files.download(zip_path)